# Customer Churn Prediction Using Artificial Neural Networks

**Project type:** Binary classification  
**Goal:** Predict whether a bank customer is likely to leave the bank (`Exited = 1`) or stay (`Exited = 0`).

## 1. Business understanding
Customer churn affects revenue and long-term customer relationships. A predictive model can help a bank identify customers who may leave and prioritize retention activities.

## 2. Dataset
This project uses the **Churn_Modelling.csv** dataset. It contains customer information such as credit score, geography, gender, age, tenure, account balance, number of products, credit-card status, activity status, estimated salary, and the target column `Exited`.

Place the CSV file inside a folder named `data` beside this notebook:
`data/Churn_Modelling.csv`

## 3. Project workflow
1. Load and inspect the data.
2. Check duplicates and target distribution.
3. Remove identifier columns.
4. Encode categorical variables.
5. Split the data into training and testing sets.
6. Standardize the features.
7. Build and train an Artificial Neural Network (ANN).
8. Evaluate predictions using classification metrics and plots.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

## 4. Load the dataset
Update the path only if your dataset is stored somewhere else.

In [ ]:
DATA_PATH = "data/Churn_Modelling.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Create a 'data' folder and place Churn_Modelling.csv inside it."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 5. Exploratory data inspection

In [ ]:
df.info()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("\\nTarget distribution:")
print(df["Exited"].value_counts())
print("\\nGeography distribution:")
print(df["Geography"].value_counts())

### Target variable
- `0`: Customer stayed with the bank.
- `1`: Customer exited the bank.

The original dataset contains 10,000 records. The target classes are imbalanced, so accuracy should not be considered alone; precision, recall, and F1-score are also important.

## 6. Data preparation
Identifier fields are removed because they do not provide useful customer-behaviour information for this prediction task. Categorical columns are converted into numeric dummy variables.

In [ ]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

df_encoded = pd.get_dummies(
    df,
    columns=["Geography", "Gender"],
    drop_first=True,
    dtype=int
)

display(df_encoded.head())

## 7. Split features and target
An 80:20 train-test split is used. `stratify=y` preserves the target-class proportions in both sets.

In [ ]:
X = df_encoded.drop(columns=["Exited"])
y = df_encoded["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

## 8. Feature scaling
StandardScaler is fitted only on the training data to prevent information leakage. The same fitted scaler transforms the test data.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 9. Build the Artificial Neural Network
The network uses two hidden Dense layers with ReLU activation and one output neuron with sigmoid activation. Sigmoid returns a probability between 0 and 1 for the positive class.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(11, activation="relu"),
    tf.keras.layers.Dense(11, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.summary()

## 10. Compile and train the model
Binary cross-entropy is used because this is a binary classification problem. A validation split monitors performance on data not used for weight updates.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.20,
    verbose=1
)

## 11. Training performance

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Model evaluation
The sigmoid output is converted into a class label using a 0.5 threshold. The threshold can be adjusted based on the business cost of missing a customer who is likely to churn.

In [ ]:
y_prob = model.predict(X_test_scaled).ravel()
threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stayed (0)", "Exited (1)"]
)
display.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 13. Conclusion

In this project, an Artificial Neural Network was developed to predict customer churn using customer demographics and banking-related attributes. The workflow covered data inspection, removal of identifier columns, categorical encoding, train-test splitting, feature standardization, neural-network training, and model evaluation.

The final model is evaluated using accuracy, precision, recall, F1-score, and a confusion matrix. Because the dataset contains fewer customers who exited than customers who stayed, **recall for the churn class (`Exited = 1`) is especially important**: a model that misses customers likely to leave may be less useful to the bank even if its overall accuracy appears high.

The model can serve as a starting point for customer-retention analysis. Further improvements could include comparing the ANN with Logistic Regression and Random Forest, tuning the classification threshold, handling class imbalance, and using cross-validation and hyperparameter tuning. The model's predictions should be validated before being used for real business decisions.

**Key learning:** This project provided practical experience with an end-to-end binary classification workflow and using TensorFlow/Keras to build and evaluate an ANN.

## 14. How to run this project
1. Download or clone the project repository.
2. Install the packages listed in `requirements.txt`.
3. Place `Churn_Modelling.csv` in the `data/` folder.
4. Open this notebook in Jupyter Notebook or JupyterLab.
5. Run the cells from top to bottom.

> **Note:** The dataset is not bundled in this notebook. Add it to the `data/` folder locally, and follow its original license/usage terms when sharing the repository.